In [5]:
import matplotlib.pyplot as plt
import pandas as pd
import os

In [ ]:
root_dir = "/Users/emmanuelle.coutu-nadeau/Library/Mobile Documents/com~apple~CloudDocs/UdeM/MSc Psycho/LABO NED - Personal Drive/Code/GENiAL/"
data_dir = os.path.join(root_dir, 'Data/Final/GENIAL-DB-preprocessed.csv')

# Debugging: Check if the file exists
if not os.path.exists(data_dir):
    raise FileNotFoundError(f"File not found: {data_dir}")

# Load the CSV file
df = pd.read_csv(data_dir)

ParserError: Error tokenizing data. C error: Expected 225 fields in line 165, saw 229


## Demographics

In [ ]:
family_member_groups = ['Proband', 'Sibling', 'Mother', 'Father', 'Child', 'Other']

# Create family_member_type histograms
plt.figure(figsize=(8, 5))
plt.hist(df['family_member_type'].dropna(), bins=10, edgecolor='black', alpha=0.7)
plt.title(f'Distribution of Family Member Type')
plt.xlabel('Family Member Type')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.savefig(f'Output/Q1K-Demographics/family_member_type_distribution.png')
plt.clf()

# Create AGE histograms for each family_member group
for group in family_member_groups:
    subset = df[df['family_member_type'] == group]
    
    mean_age = subset['Age at EEG (years)'].mean()
    median_age = subset['Age at EEG (years)'].median()
    min_age = subset['Age at EEG (years)'].min()
    max_age = subset['Age at EEG (years)'].max()

    plt.figure(figsize=(8, 5))
    plt.hist(subset['Age at EEG (years)'].dropna(), bins=10, edgecolor='black', alpha=0.7)
    plt.title(f'Age Distribution for {group}')
    plt.xlabel('Age')
    plt.ylabel('Frequency')
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    # Add legend with statistical information
    stats_text = f"Mean: {mean_age:.2f}\nMedian: {median_age:.2f}\nMin: {min_age}\nMax: {max_age}"
    plt.legend([stats_text], loc='upper right', fontsize=10, frameon=True)


    plt.savefig(f'Output/Q1K-Demographics/age_distribution_{group}.png')
    plt.clf()

# Create GENDER histograms for each family_member group
for group in family_member_groups:
    subset = df[df['family_member_type'] == group]

    # Count occurrences of each gender category
    gender_counts = subset['Sex at birth:'].value_counts()


    # Create pie chart
    plt.figure(figsize=(8, 5))
    wedges, texts, autotexts = plt.pie(
        gender_counts, 
        labels=gender_counts.index,  # Keep only category names as labels
        autopct=lambda p: f'{p:.1f}% ({int(p * sum(gender_counts) / 100)})',  # Show percentage and absolute value
        startangle=90, 
        wedgeprops={'edgecolor': 'black'}
    )

    plt.title(f'Sex at Birth Distribution for {group}')

    # Add legend
    plt.legend(wedges, gender_counts.index, title="Sex at Birth", loc="upper right")

    # Save the figure
    plt.savefig(f'Output/Q1K-Demographics/gender_distribution_{group}.png')
    plt.clf()  # Clear the figure after saving

In [ ]:
# Filter relevant genetic status categories
genetic_status_filtered = df[df['Genetic Status'].isin(['Normal', 'Abnormal', 'VUS', 'NaN'])]

# Count occurrences of each Genetic Status
genetic_status_counts = genetic_status_filtered['Genetic Status'].value_counts()

# Count occurrences of Genetic Abnormality Type within "Abnormal" and "VUS"
abnormal_vus_counts = genetic_status_filtered[
    genetic_status_filtered['Genetic Status'].isin(['Abnormal', 'VUS'])
].groupby(['Genetic Status', 'Genetic Abnormality Type']).size().unstack(fill_value=0)

# Create stacked bar chart
fig, ax = plt.subplots(figsize=(12, 8))

# Plot total count of Genetic Status
ax.bar(genetic_status_counts.index, genetic_status_counts.values, color='gray', alpha=0.5, label="Total Count")

# Overlay with breakdown of Abnormality Types within "Abnormal" and "VUS"
bottoms = pd.Series(0, index=abnormal_vus_counts.index)  # Initialize bottom positions

for abnormality in abnormal_vus_counts.columns:
    ax.bar(abnormal_vus_counts.index, abnormal_vus_counts[abnormality], label=abnormality, bottom=bottoms)
    bottoms += abnormal_vus_counts[abnormality]  # Update bottom positions for stacking

# Labels and title
ax.set_ylabel("Count")
ax.set_title("Genetic Status and Abnormality Type Distribution")
ax.legend(title="Genetic Abnormality Type", bbox_to_anchor=(1.05, 1), loc="upper right")
plt.xticks(rotation=45)

# Save figure
plt.savefig(f'Output/Q1K-Demographics/genetics.png')
plt.clf()  # Clear the figure after saving
